# Tuned architectures from the earlier thesis (Brent and WTI, 5, 22 and 66 days)

Runs `experiments/vol_dl_arch.py` from the `oilfc-residual` Kaggle dataset, as pre-registered in
`docs/adr/0007-deep-learning-architectures-tuned.md`. Six cells (two assets, three horizons); each trains N-HiTS,
TiDE, iTransformer, TCN, TFT and Informer (hyperparameters tuned before 2012, read from `results/dl_arch/hparams.json`)
as QLIKE-trained bounded corrections to the headline benchmark, plus a ridge control, and writes its results as soon as
it finishes. Set **Accelerator = GPU**, run the cells in order, and download `dl_arch_results.zip` from the Output
panel. No internet is needed.


In [1]:
import glob
import os
import shutil
import subprocess
import sys
import zipfile

import torch

print(
    "torch",
    torch.__version__,
    "| cuda:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

torch 2.11.0+cu130 | cuda: True Tesla T4


In [2]:
# 1. Locate the dataset and copy the code to /kaggle/working
WORK = "/kaggle/working"
hits = glob.glob("/kaggle/input/**/experiments/vol_dl_arch.py", recursive=True)
if hits:
    base = os.path.dirname(os.path.dirname(hits[0]))
else:
    zp = glob.glob("/kaggle/input/**/oilfc-residual.zip", recursive=True)[0]
    base = f"{WORK}/_dataset"
    zipfile.ZipFile(zp).extractall(base)
PROJ = f"{WORK}/proj"
shutil.copytree(base, PROJ, dirs_exist_ok=True)
os.chdir(PROJ)
print(os.listdir(PROJ))

['results', 'data', 'README.txt', 'experiments', 'src']


In [3]:
# 2. Helper and smoke test (a few minutes): 13 monthly splits, 1 seed, 5 epochs. Fix any error here before the full runs.
def run(args):
    p = subprocess.Popen(
        [sys.executable, "-u", "experiments/vol_dl_arch.py", *args, "--out", f"{WORK}/results"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        if "warn" not in line.lower():
            print(line, end="")
    assert p.wait() == 0, "run failed"


run(["--smoke", "--device", "auto"])

device=cuda seeds=(0,) epochs=5 h=22 asset=brent
13 monthly splits, 792 window columns
ridge_window: 1s, mean alpha 0.46
nhits: 10s, mean alpha 0.00
tide: 1s, mean alpha 0.83
itransformer: 4s, mean alpha 0.00
tcn: 8s, mean alpha 0.00
tft: 3s, mean alpha 0.08
informer: 2s, mean alpha 0.92
                n   qlike  qlike_benchmark  gain_vs_benchmark_pct  dm_p_better  mean_alpha  share_alpha_zero  seconds
variant                                                                                                              
ridge_window  270  0.1702           0.1769                 3.8040       0.2513      0.4611            0.0778   0.8898
nhits         270  0.1769           0.1769                 0.0000          NaN      0.0000            1.0000  10.0369
tide          270  0.1837           0.1769                -3.8234       0.5938      0.8300            0.0778   1.2247
itransformer  270  0.1769           0.1769                 0.0000          NaN      0.0000            1.0000   4.0238
tcn

In [4]:
# 3. Brent, 5 and 22 days (5 seeds)
for h in ("22", "5"):
    run(["--asset", "brent", "--h", h, "--seeds", "5", "--device", "auto"])

device=cuda seeds=(0, 1, 2, 3, 4) epochs=80 h=22 asset=brent
162 monthly splits, 792 window columns
ridge_window: 12s, mean alpha 0.53
nhits: 96s, mean alpha 0.25
tide: 103s, mean alpha 0.60
itransformer: 202s, mean alpha 0.48
tcn: 127s, mean alpha 0.45
tft: 329s, mean alpha 0.58
informer: 227s, mean alpha 0.46
                 n   qlike  qlike_benchmark  gain_vs_benchmark_pct  dm_p_better  mean_alpha  share_alpha_zero   seconds
variant                                                                                                                
ridge_window  3396  0.2491           0.2516                 1.0112       0.3620      0.5330            0.2589   11.5429
nhits         3396  0.2495           0.2516                 0.8345       0.3924      0.2489            0.2952   95.6023
tide          3396  0.2418           0.2516                 3.8767       0.2366      0.6031            0.1486  102.7197
itransformer  3396  0.2400           0.2516                 4.6176       0.1795      0.

In [5]:
# 4. WTI, 5 and 22 days
for h in ("22", "5"):
    run(["--asset", "wti", "--h", h, "--seeds", "5", "--device", "auto"])

device=cuda seeds=(0, 1, 2, 3, 4) epochs=80 h=22 asset=wti
162 monthly splits, 792 window columns
ridge_window: 9s, mean alpha 0.52
nhits: 83s, mean alpha 0.25
tide: 100s, mean alpha 0.41
itransformer: 167s, mean alpha 0.52
tcn: 127s, mean alpha 0.27
tft: 317s, mean alpha 0.47
informer: 227s, mean alpha 0.47
                 n   qlike  qlike_benchmark  gain_vs_benchmark_pct  dm_p_better  mean_alpha  share_alpha_zero   seconds
variant                                                                                                                
ridge_window  3312  0.2822           0.3111                 9.2730       0.1233      0.5247            0.1854    8.9538
nhits         3312  0.3081           0.3111                 0.9547       0.3869      0.2533            0.3320   83.2155
tide          3312  0.3116           0.3111                -0.1776       0.5199      0.4142            0.2600   99.6088
itransformer  3312  0.3004           0.3111                 3.4418       0.2782      0.518

In [6]:
# 5. 66 days, both assets
for asset in ("brent", "wti"):
    run(["--asset", asset, "--h", "66", "--seeds", "5", "--device", "auto"])

device=cuda seeds=(0, 1, 2, 3, 4) epochs=80 h=66 asset=brent
162 monthly splits, 792 window columns
ridge_window: 10s, mean alpha 0.59
nhits: 97s, mean alpha 0.44
tide: 113s, mean alpha 0.59
itransformer: 185s, mean alpha 0.59
tcn: 128s, mean alpha 0.47
tft: 323s, mean alpha 0.56
informer: 233s, mean alpha 0.34
                 n   qlike  qlike_benchmark  gain_vs_benchmark_pct  dm_p_better  mean_alpha  share_alpha_zero   seconds
variant                                                                                                                
ridge_window  3352  0.5889           0.6201                 5.0281       0.1117      0.5946            0.0737    9.7763
nhits         3352  0.5936           0.6201                 4.2712       0.2463      0.4440            0.2952   96.5704
tide          3352  0.5313           0.6201                14.3083       0.1665      0.5921            0.2224  112.8478
itransformer  3352  0.5777           0.6201                 6.8391       0.1714      0.

In [7]:
# 6. Package whatever has finished for download (safe to run after any cell)
zp = f"{WORK}/dl_arch_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{WORK}/results/*"):
        z.write(f, os.path.basename(f))
print("ready:", zp, sorted(os.path.basename(f) for f in glob.glob(f"{WORK}/results/*")))

ready: /kaggle/working/dl_arch_results.zip ['dl_arch_brent_22d.csv', 'dl_arch_brent_22d.json', 'dl_arch_brent_22d_smoke.csv', 'dl_arch_brent_22d_smoke.json', 'dl_arch_brent_5d.csv', 'dl_arch_brent_5d.json', 'dl_arch_brent_66d.csv', 'dl_arch_brent_66d.json', 'dl_arch_predictions_brent_22d.csv', 'dl_arch_predictions_brent_22d_smoke.csv', 'dl_arch_predictions_brent_5d.csv', 'dl_arch_predictions_brent_66d.csv', 'dl_arch_predictions_wti_22d.csv', 'dl_arch_predictions_wti_5d.csv', 'dl_arch_predictions_wti_66d.csv', 'dl_arch_wti_22d.csv', 'dl_arch_wti_22d.json', 'dl_arch_wti_5d.csv', 'dl_arch_wti_5d.json', 'dl_arch_wti_66d.csv', 'dl_arch_wti_66d.json']
